# 단계 3: 데이터 전처리 및 표준화

**목표**: 금융 데이터 컬럼명 표준화 및 분기별 집계

**입력**: `data/processed/[기업명]_with_scores.csv`

**출력**: `data/processed/final_dataset_standardized.csv`

## 1. 필수 라이브러리 임포트

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

from src.data_processor import rename_financial_columns, select_final_columns, fill_missing_values
import pandas as pd
import numpy as np
import yaml

print("✅ 모든 라이브러리 임포트 완료")

## 2. 설정 로드

In [ ]:
config_path = os.path.join(os.path.dirname(__file__), '..', 'config.yaml')

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("✅ 설정 로드 완료")

## 3. 데이터 로드

In [ ]:
# 데이터 경로
COMPANY_NAME = '삼성전자'
input_path = os.path.join(config['DATA_PATHS']['processed_data'], f"{COMPANY_NAME}_with_scores.csv")

try:
    df = pd.read_csv(input_path, encoding='utf-8')
    print(f"✅ 데이터 로드 완료")
    print(f"   행 수: {len(df)}")
    print(f"   컬럼: {df.columns.tolist()}")
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {input_path}")
    print("   먼저 02_llm_confident.ipynb을 실행해주세요.")
except Exception as e:
    print(f"❌ 데이터 로드 실패: {e}")

## 4. 결측값 처리

In [ ]:
print("📊 결측값 진단:")
print(df.isnull().sum())

# 결측값 채우기
df_cleaned = fill_missing_values(df, method='mean')

print("\n✅ 결측값 처리 완료")
print(df_cleaned.isnull().sum())

## 5. 분기별(Quarter) 정보 추가

In [ ]:
# date 컬럼을 datetime으로 변환
if 'date' in df_cleaned.columns:
    df_cleaned['date'] = pd.to_datetime(df_cleaned['date'], format='%Y%m%d', errors='coerce')
    
    # 분기 정보 추가
    df_cleaned['year'] = df_cleaned['date'].dt.year
    df_cleaned['quarter'] = df_cleaned['date'].dt.quarter
    df_cleaned['year_quarter'] = df_cleaned['year'].astype(str) + '-Q' + df_cleaned['quarter'].astype(str)
    
    print("✅ 분기 정보 추가 완료")
    print(f"\n분기별 레코드 수:")
    print(df_cleaned['year_quarter'].value_counts().sort_index().head(10))
else:
    print("⚠️  'date' 컬럼이 없습니다.")

## 6. 주요 컬럼 표준화

In [ ]:
# 컬럼 표준화 매핑
rename_map = {
        'confidence_score': '확신도점수',
        'bullish_outlook': '강세전망점수',
        'composite_score': '종합의견점수'
}

df_renamed = rename_financial_columns(df_cleaned, rename_map)

print("✅ 컬럼명 표준화 완료")
print(f"   변경된 컬럼: {list(rename_map.values())}")

## 7. 분기별 집계

In [ ]:
# 분기별 평균 및 표준편차 계산
quarterly_agg = df_renamed.groupby('year_quarter').agg({
    '확신도점수': ['mean', 'std', 'count'],
    '강세전망점수': ['mean', 'std'],
    '종합의견점수': ['mean', 'std']
}).round(2)

quarterly_agg.columns = ['_'.join(col).strip() for col in quarterly_agg.columns.values]

print("✅ 분기별 집계 완료")
print(f"\n분기별 통계:")
print(quarterly_agg.head(10))

## 8. 최종 데이터 저장

In [ ]:
# 최종 데이터 저장
output_path = os.path.join(config['DATA_PATHS']['processed_data'], 'final_dataset_standardized.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_renamed.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 저장 완료: {output_path}")
print(f"\n최종 데이터 구조:")
print(df_renamed.head(3))

## 9. 데이터 요약

In [ ]:
print("📊 최종 데이터 요약:")
print(f"  총 레코드 수: {len(df_renamed)}")
print(f"  분기 수: {df_renamed['year_quarter'].nunique()}")
print(f"  컬럼 수: {len(df_renamed.columns)}")
print(f"\n주요 컬럼:")
for col in ['확신도점수', '강세전망점수', '종합의견점수']:
    if col in df_renamed.columns:
        print(f"  {col}: 평균={df_renamed[col].mean():.2f}, 최소={df_renamed[col].min():.2f}, 최대={df_renamed[col].max():.2f}")